# [8.4] Circuit Tracing with Attribution Graphs - Solutions

This notebook runs the reference implementation, visible tests, committed CUDA report inspection, and live CUDA preflight.

<details>
<summary>Expected output</summary>

All section-local tests should pass, the committed report should show `position_5 -> position_5`, and the live CUDA run should reproduce the same graph checks.

</details>

<details>
<summary>Help - claim boundary</summary>

This is a GT-1 residual-position graph preflight, not full sparse-feature/transcoder circuit tracing.

</details>

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

chapter = "chapter8_automated_circuits"
section = "part4_circuit_tracing_attribution_graphs"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_circuit_tracing_attribution_graphs.tests as tests
import part4_circuit_tracing_attribution_graphs.utils as utils
from chapter8_automated_circuits.exercises.part4_circuit_tracing_attribution_graphs import solutions

GT_TIER = "GT-1"
EXERCISE_ID = "8_4_circuit_tracing_with_attribution_graphs"
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-2 minutes for the CUDA preflight"
REQUIRES_GPU = True
DIFFICULTY = 4
IMPORTANCE = 4

In [ ]:
tests.test_edge_attribution_scores_forms_position_edge_matrix(solutions.edge_attribution_scores)
tests.test_edge_attribution_scores_rejects_degenerate_inputs(solutions.edge_attribution_scores)
tests.test_build_local_attribution_graph_keeps_top_directed_edges(solutions.build_local_attribution_graph)
tests.test_build_local_attribution_graph_rejects_bad_nodes_or_scores(solutions.build_local_attribution_graph)
tests.test_graph_metric_report_measures_explained_fraction(solutions.graph_metric_report)
tests.test_metric_reports_reject_nonfinite_or_negative_thresholds(
    solutions.graph_metric_report,
    solutions.path_perturbation_report,
    solutions.alternative_graph_baseline_report,
)
tests.test_path_perturbation_and_alternative_baseline_reports(
    solutions.path_perturbation_report,
    solutions.alternative_graph_baseline_report,
)
tests.test_counterfactual_summary_report_checks_direction(solutions.graph_summary_counterfactual_report)
tests.test_top_attribution_path_recovers_multi_hop_chain(solutions.top_attribution_path)
tests.test_top_attribution_path_rejects_invalid_graphs(solutions.top_attribution_path)
tests.test_notebook_contract(solutions.run_smoke_test)

utils.print_report("CPU circuit-tracing contract", solutions.run_smoke_test())

## Signature Result

<details>
<summary>Expected output</summary>

The committed report should have `preflight_passed=True`, graph edge `position_5 -> position_5`, explained fraction `1.0`, and peak VRAM below 1 GB.

</details>

<details>
<summary>Common bug</summary>

Do not treat the one-edge residual-position graph as full sparse-feature/transcoder tracing.

</details>

<details>
<summary>Solution</summary>

Read the committed `verification_report.json`, assert the graph checks, and plot the accepted metric checks.

</details>

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"] and report["tests_passed"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["graph_edge_source"] == "position_5"
assert gpu["graph_edge_target"] == "position_5"
assert gpu["num_graph_edges"] == 1
assert gpu["explains_target_metric"]
assert gpu["top_path_survives_test"]
assert gpu["alternative_baseline_fails"]
assert gpu["predicts_counterfactual"]
assert gpu["peak_vram_gb"] <= 24.0

fig, ax = plt.subplots(figsize=(7.5, 3.2))
labels = ["explained", "path drop", "alt margin", "counterfact"]
values = [
    gpu["explained_fraction"],
    gpu["path_metric_drop"] / gpu["clean_corrupt_gap"],
    gpu["alternative_baseline_margin"] / gpu["clean_corrupt_gap"],
    abs(gpu["counterfactual_observed_delta"]) / gpu["clean_corrupt_gap"],
]
ax.bar(labels, values, color=["#2563eb", "#10b981", "#f97316", "#7c3aed"])
ax.set_ylim(0, 1.15)
ax.set_ylabel("normalized check value")
ax.set_title("8.4 committed graph-tracing checks")
for i, value in enumerate(values):
    ax.text(i, value + 0.03, f"{value:.2f}", ha="center", fontsize=10)
fig.tight_layout()
plt.show()

utils.print_report(
    "Committed CUDA circuit-tracing report",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "device": gpu["device"],
        "edge": f'{gpu["graph_edge_source"]} -> {gpu["graph_edge_target"]}',
        "edge_score": round(gpu["graph_edge_score"], 4),
        "explained_fraction": gpu["explained_fraction"],
        "path_drop": round(gpu["path_metric_drop"], 4),
        "alternative_margin": round(gpu["alternative_baseline_margin"], 4),
        "counterfactual_delta": round(gpu["counterfactual_observed_delta"], 4),
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)

## Live CUDA Verification

<details>
<summary>Expected output</summary>

The live run should load `gelu-1l`, use CUDA 13.2 under torch `2.12.1+cu132`, and reproduce the one-edge final-position graph checks.

</details>

<details>
<summary>Help - why run this after reading the committed report?</summary>

The committed report proves the release artifact; the live run proves the current environment can still execute the real model path.

</details>

In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return solutions.run_full_experiment(max_vram_gb=max_vram_gb)


live_gpu = run_full_experiment(max_vram_gb=24.0)
assert live_gpu["preflight_passed"]
assert live_gpu["graph_edge_source"] == "position_5"
assert live_gpu["graph_edge_target"] == "position_5"
assert live_gpu["explains_target_metric"]
assert live_gpu["top_path_survives_test"]
assert live_gpu["alternative_baseline_fails"]
assert live_gpu["predicts_counterfactual"]
assert live_gpu["peak_vram_gb"] <= 24.0
utils.print_report(
    "Live CUDA circuit-tracing preflight",
    {
        "torch": live_gpu["torch_version"],
        "cuda": live_gpu["cuda_version"],
        "device": live_gpu["device"],
        "edge": f'{live_gpu["graph_edge_source"]} -> {live_gpu["graph_edge_target"]}',
        "edge_score": round(live_gpu["graph_edge_score"], 4),
        "explained_fraction": live_gpu["explained_fraction"],
        "path_drop": round(live_gpu["path_metric_drop"], 4),
        "alternative_margin": round(live_gpu["alternative_baseline_margin"], 4),
        "counterfactual_delta": round(live_gpu["counterfactual_observed_delta"], 4),
        "peak_vram_gb": round(live_gpu["peak_vram_gb"], 4),
    },
)

## Limitations and Further Research

<details>
<summary>Limitations</summary>

The result is one prompt pair, one hook, residual-position nodes, and a one-edge graph. It does not establish full circuit tracing.

</details>

<details>
<summary>Further Research</summary>

Replace residual-position nodes with attention heads, SAE features, or transcoder features; add OOD prompt suites; and pre-register graph-summary counterfactuals before interventions.

</details>